# FMCW inference
Process one complex chirp with the supplied epoch-120 checkpoint.
No training or calibration occurs. The supplied export includes its validation-selected
threshold (0.6254660674). The example belongs to training data.


## 1. Project and dependencies


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys
try:
    from google.colab import drive
except ImportError:
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT = next((p for p in candidates if (p/'training/fmcw_training').is_dir()), Path.cwd())
else:
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/fmcw-mitigation-detection')
# Change PROJECT above if your folder is elsewhere.
assert (PROJECT/'training/fmcw_training').is_dir(), 'Set PROJECT to the repository folder.'
sys.path.insert(0, str(PROJECT/'training'))
missing = [name for name in ['torch', 'numpy', 'h5py', 'matplotlib', 'tqdm']
           if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r',
                           str(PROJECT/'training/requirements.txt')])
print('Project:', PROJECT)


## 2. Load the checkpoint and example
Radar values use SI units. Change `WEIGHTS`, `DEVICE` or `THRESHOLD` as needed.
Setting `THRESHOLD=None` uses a saved threshold, or 0.5 if the checkpoint has none.


In [ ]:
import json
import numpy as np
from predictor import Predictor
WEIGHTS = PROJECT/'training/weights/epoch_0120.pt'
DEVICE = 'cpu'  # Use 'cuda' when available.
THRESHOLD = None
with np.load(PROJECT/'training/example_training_scene.npz', allow_pickle=False) as example:
    observed = example['y'].copy()
    radar = json.loads(str(example['radar_json']))
predictor = Predictor(WEIGHTS, device=DEVICE, threshold=THRESHOLD)
result = predictor(observed, **radar, diagnostic=True)
print('Epoch:', predictor.epoch)
print('Threshold:', result['threshold'], 'Source:', result['threshold_source'])
print('Detected ranges (m):', result['detected_m'])


## 3. Reconstruction and presence scores


In [ ]:
import matplotlib.pyplot as plt
r = result['distance_m']
valid = result['valid']
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(r[valid], abs(result['complex_reconstruction'][valid]))
axes[0].set_ylabel('Reconstructed FFT amplitude (relative)')
axes[1].plot(r[valid], result['presence'][valid])
axes[1].axhline(result['threshold'], color='grey', linestyle='--')
axes[1].set_ylabel('Presence score')
for ax in axes:
    ax.set_xlabel('Range (m)')
    ax.grid(alpha=.2)
plt.tight_layout()
plt.show()
